In [0]:
import requests
import uuid
from datetime import datetime
from dateutil.relativedelta import relativedelta

# Fonctions

In [0]:
def get_data(url, params, filename, date):
    response = requests.get(url, params=params)
    
    if response.status_code == 200:
        data = response.content

        dbutils.fs.put(
            f"/Volumes/transport/bronze/files/{filename}-{date}.csv",
            data.decode("utf-8-sig"),
            overwrite=True
        )
        print("Données récupérées avec succès")
        return True
    else:
        print(f"Erreur lors de la récupération des données pour : {response.status_code}")
        print(response.text)
        return False

# Variables

In [0]:
# Généré une fois au début du pipeline
runid = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
# Revenir deux mois plus tot
date = (datetime.now() - relativedelta(months=2)).strftime("%Y-%m")

dbutils.jobs.taskValues.set(key="runid", value=runid)
dbutils.jobs.taskValues.set(key="date", value=date)

print(f"Run ID : {runid}")
print(f"Run date : {date}")

In [0]:
params = {
    "delimiter": ";",
    "lang": "fr",
    "timezone": "Europe/Berlin",
    "use_labels": "true",
    "refine": f'date:"{date}"',
}

# Fichiers de régularités brutes

In [0]:
url_ter = "https://ressources.data.sncf.com/api/explore/v2.1/catalog/datasets/regularite-mensuelle-ter/exports/csv"
url_tgv= "https://ressources.data.sncf.com/api/explore/v2.1/catalog/datasets/regularite-mensuelle-tgv-aqst/exports/csv"
url_transilien = "https://ressources.data.sncf.com//api/explore/v2.1/catalog/datasets/ponctualite-mensuelle-transilien/exports/csv"

liste_url = [
    url_ter,
    url_tgv,
    url_transilien
    ]

liste_filename = [
    "regularite_mensuelle_ter",
    "regularite_mensuelle_tgv",
    "regularite_mensuelle_transilien"
    ]

for i, j in zip(liste_url, liste_filename):
    get_data(i, params, j, date)

# Fichier gares de voyageurs 

In [0]:
url_gare_voyageur = "https://ressources.data.sncf.com/api/explore/v2.1/catalog/datasets/gares-de-voyageurs/exports/csv"
url_toutes_gares = "https://ressources.data.sncf.com/api/explore/v2.1/catalog/datasets/liste-des-gares/exports/csv"

params = {
    "delimiter": ";",
    "lang": "fr",
    "timezone": "Europe/Berlin",
    "use_labels": "true",
}

get_data(url_gare_voyageur, params, "gares_de_voyageurs", date)
get_data(url_toutes_gares, params, "liste_des_gares", date)